# P2_81 — M4 protocol sensitivity (NOT YET EXECUTED)
Run all cells on a Colab GPU. At the end, send back `P2_81_M4_PROTOCOL_SENSITIVITY_RESULTS.zip`.


In [ ]:
%pip -q install -U "transformers>=4.57,<5" accelerate bitsandbytes safetensors pandas openpyxl "scikit-learn>=1.4"


In [ ]:
#!/usr/bin/env python3
"""P2_81 M4 protocol-sensitivity audit (NOT YET EXECUTED).

Run in a GPU environment after installing:
  pip install 'transformers>=4.57,<5' accelerate bitsandbytes safetensors pandas openpyxl 'scikit-learn>=1.4'

On Colab, the script assumes /content/drive/MyDrive/P2/{input,results}.
It reruns Condition C only under answer-code permutations and two equivalent prompt templates.
"""
from pathlib import Path
import os, sys, json, hashlib, zipfile
import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = Path('/content/drive/MyDrive/P2') if IN_COLAB else Path('/content/P2')
INPUT_DIR = PROJECT_DIR/'input'; RESULTS_DIR = PROJECT_DIR/'results'
OUT_DIR = RESULTS_DIR/'P2_81_M4_PROTOCOL_SENSITIVITY'
LOCAL_CACHE = Path('/content/p281_m4_sensitivity_cache')
for d in [INPUT_DIR, RESULTS_DIR, OUT_DIR, LOCAL_CACHE]: d.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(LOCAL_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(LOCAL_CACHE/'transformers')

GOLD_NAME='P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx'
BASE_NAME='P2_CPU_Baselines_v1.0.xlsx'
EXPECTED={
    GOLD_NAME:'ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65',
    BASE_NAME:'4502e71bd6d1b6d2941b0b10650caaa433d7188bf958017d2fc646d328a31163',
}
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for ch in iter(lambda:f.read(1024*1024),b''): h.update(ch)
    return h.hexdigest()
for n,e in EXPECTED.items():
    p=INPUT_DIR/n
    assert p.exists(), f'Missing {p}'
    o=sha256(p); print(n,o); assert o==e, 'Frozen input hash mismatch.'

gold=pd.read_excel(INPUT_DIR/GOLD_NAME,sheet_name='FINAL_GOLD')
gold_by_case=gold.set_index('Case_ID',drop=False)
oof={t:pd.read_excel(INPUT_DIR/BASE_NAME,sheet_name=f'OOF_{t}') for t in ['T1','T2','T3','T4']}
assert {t:len(x) for t,x in oof.items()}=={'T1':720,'T2':474,'T3':460,'T4':474}

MODEL_ID='Qwen/Qwen3-8B'
MODEL_REVISION='e8bbd8252970581ea5b08b6a5b3e668adaf3161a'
BATCH_SIZE=4
BOOT_REPS=2000
SEED=20260908
assert torch.cuda.is_available(), 'Select a GPU runtime.'
print('GPU:',torch.cuda.get_device_name(0),'transformers:',transformers.__version__)
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION)
tokenizer.padding_side='left'
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,quantization_config=quant,device_map='auto')
model.eval()
CODE_ID={}
for c in ['A','B','C']:
    ids=tokenizer(c,add_special_tokens=False)['input_ids']
    print(c,ids); assert len(ids)==1
    CODE_ID[c]=ids[0]

def serialise_C(g):
    p=str(g['Prompt_Text'] if pd.notna(g['Prompt_Text']) else '').strip()
    s=str(g['Source_Context'] if pd.notna(g['Source_Context']) else '').strip()
    o=str(g['Model_Output'] if pd.notna(g['Model_Output']) else '').strip()
    if str(g['Source_Available']).lower()=='yes' and s:
        return f"Task:\n{p}\n\nAuthorised source/context:\n{s}\n\nResponse:\n{o}"
    return f"Task:\n{p}\n\nResponse:\n{o}"

MEANINGS={
'T1':{0:'no_failure: no identifiable failure under the supplied evidence.',
      1:'failure: at least one identifiable semantic-coherence or authorised-grounding failure.'},
'T2':{0:'semantic_only: the failure is meaning-level but does not require unsupported/fabricated content or conflict with an authorised visible anchor.',
      1:'grounding_involved: the failure includes overlap with unsupported/fabricated content or authorised-anchor conflict, including hallucination-only cases.'},
'T3':{0:'Core_SCF: topic/QUD shift, causal-chain failure, response-goal misalignment, discourse-structure failure, local-to-global breakdown, referential instability, or term-sense instability.',
      1:'OVR: self-contradiction, visible-context inconsistency, entity/role/attribute misattribution, temporal/world-state inconsistency, reasoning-answer mismatch, or mixed-context synthesis failure.'},
'T4':{1:'severity 1 (Minor): localized disruption with limited effect on the main answer.',
      2:'severity 2 (Moderate): materially weakens an important task-relevant relation or component.',
      3:'severity 3 (Severe): undermines the main answer, reverses meaning, contradicts the prompt/source, or makes the answer unreliable for the task.'},
}
PERMS={
'T1':{'identity':{'A':0,'B':1},'swap':{'A':1,'B':0}},
'T2':{'identity':{'A':0,'B':1},'swap':{'A':1,'B':0}},
'T3':{'identity':{'A':0,'B':1},'swap':{'A':1,'B':0}},
'T4':{'identity':{'A':1,'B':2,'C':3},'cycle_231':{'A':2,'B':3,'C':1},'cycle_312':{'A':3,'B':1,'C':2}},
}
SYSTEM_P1=("You are a conservative semantic-coherence evaluator.\n"
           "Apply only the supplied target rubric to the visible text.\n"
           "Do not infer hidden facts. Do not output an explanation.\n"
           "Your decision space is closed: select exactly one of the permitted answer codes.")
SYSTEM_P2=("Classify the visible case using only the criterion stated below.\n"
           "Use no outside knowledge and do not infer evidence that is not visible.\n"
           "Return a single permitted code only; do not provide reasoning or any additional text.")
P1_INTRO={
'T1':'Decide only whether the response contains an identifiable semantic-coherence or authorised-grounding failure.',
'T2':'The case is already known to contain a failure. Decide whether authorised grounding is involved.',
'T3':'The case belongs to one of two mechanism families.',
'T4':'The case is already known to contain a failure. Judge functional severity, not confidence.',
}
P1_END={'T1':'Do not classify the subtype. Choose only A or B.',
        'T2':'Choose only A or B.','T3':'Choose only A or B.','T4':'Choose only A, B, or C.'}

def rubric(target,mapping,template):
    lines=[f'{code} = {MEANINGS[target][label]}' for code,label in mapping.items()]
    if template=='P1':
        # Identity mapping reproduces the final-v3 rubric exactly; permutations change codes only.
        return P1_INTRO[target]+'\n'+'\n'.join(lines)+'\n'+P1_END[target]
    if target=='T1': intro='Determine whether the visible response has a material semantic-coherence or authorised-grounding failure.'
    elif target=='T2': intro='A failure is already present. Determine whether the visible evidence makes grounding part of that failure.'
    elif target=='T3': intro='Assign the known failure to the better-fitting mechanism family.'
    else: intro='A failure is already present. Rate its functional impact on the requested task rather than your confidence.'
    return ('Criterion: '+intro+'\n\nCode definitions:\n'+'\n'.join(lines)+
            '\n\nSelect the one code whose definition best matches the visible case.')

def make_chat(g,target,mapping,template):
    sysmsg=SYSTEM_P1 if template=='P1' else SYSTEM_P2
    user=rubric(target,mapping,template)+'\n\nVisible case:\n'+serialise_C(g)
    return tokenizer.apply_chat_template(
        [{'role':'system','content':sysmsg},{'role':'user','content':user}],
        tokenize=False,add_generation_prompt=True,enable_thinking=False)

@torch.inference_mode()
def score_batch(texts,codes):
    enc=tokenizer(texts,return_tensors='pt',padding=True,truncation=True,max_length=4096)
    enc={k:v.to(model.device) for k,v in enc.items()}
    out=model(**enc)
    logp=torch.log_softmax(out.logits[:,-1,:].float(),dim=-1)
    ids=torch.tensor([CODE_ID[c] for c in codes],device=logp.device)
    return logp[:,ids].detach().cpu().numpy()

def run_variant(target,template,perm_name,mapping):
    out_path=OUT_DIR/f'OOF_M4_SENS_{target}_{template}_{perm_name}.csv'
    rows=oof[target].copy(); codes=list(mapping.keys())
    existing={}
    if out_path.exists():
        old=pd.read_csv(out_path)
        for _,r in old.iterrows(): existing[str(r.Case_ID)]=r.to_dict()
        print('resume',target,template,perm_name,len(existing))
    pending=[r for _,r in rows.iterrows() if str(r.Case_ID) not in existing]
    records=list(existing.values())
    for start in range(0,len(pending),BATCH_SIZE):
        batch=pending[start:start+BATCH_SIZE]
        texts=[make_chat(gold_by_case.loc[str(r.Case_ID)],target,mapping,template) for r in batch]
        scores=score_batch(texts,codes)
        for r,sc in zip(batch,scores):
            j=int(np.argmax(sc)); code=codes[j]; pred=int(mapping[code]); ss=np.sort(sc)
            records.append({
                'Case_ID':str(r.Case_ID),'Prompt_ID':r.Prompt_ID,'Task_Type':r.Task_Type,
                'Fold':int(r.Fold),'y_true':int(r.y_true),'template':template,'permutation':perm_name,
                'M4_pred':pred,'M4_code':code,
                'M4_code_scores':json.dumps({c:float(v) for c,v in zip(codes,sc)}),
                'M4_score_margin':float(ss[-1]-ss[-2]) if len(ss)>1 else np.nan,
            })
        pd.DataFrame(records).drop_duplicates('Case_ID',keep='last').to_csv(out_path,index=False)
        if start % 100 == 0:
            print(target,template,perm_name,min(start+BATCH_SIZE,len(pending)),'/',len(pending))
    d=pd.read_csv(out_path).drop_duplicates('Case_ID',keep='last')
    assert len(d)==len(rows) and d.Case_ID.astype(str).nunique()==len(rows)
    assert set(d.Case_ID.astype(str))==set(rows.Case_ID.astype(str))
    assert set(d.M4_pred.astype(int)).issubset(set(mapping.values()))
    return d

all_runs={}
for target in ['T1','T2','T3','T4']:
    for template in ['P1','P2']:
        for pname,mapping in PERMS[target].items():
            print('RUN',target,template,pname)
            all_runs[(target,template,pname)]=run_variant(target,template,pname,mapping)

def macro_from_cm(cm):
    tp=np.diag(cm).astype(float); fp=cm.sum(0)-tp; fn=cm.sum(1)-tp; den=2*tp+fp+fn
    return float(np.divide(2*tp,den,out=np.zeros_like(tp),where=den!=0).mean())
def cluster_cms(df,pred,labels):
    ix={v:i for i,v in enumerate(labels)}; arr=[]
    for _,s in df.assign(_pid=df.Prompt_ID.astype(str)).groupby('_pid',sort=True):
        cm=np.zeros((len(labels),len(labels)),int)
        for y,p in zip(s.y_true.astype(int),s[pred].astype(int)): cm[ix[y],ix[p]]+=1
        arr.append(cm)
    return np.stack(arr)
def ci(df,pred,labels,reps=BOOT_REPS):
    cms=cluster_cms(df,pred,labels); rng=np.random.default_rng(SEED); vals=[]
    for _ in range(reps):
        q=rng.choice(len(cms),size=len(cms),replace=True)
        vals.append(macro_from_cm(cms[q].sum(0)))
    return np.percentile(vals,[2.5,97.5])

metric_rows=[]; dist_rows=[]; agreement_rows=[]
for target in ['T1','T2','T3','T4']:
    labels=sorted(oof[target].y_true.astype(int).unique())
    ref=all_runs[(target,'P1','identity')][['Case_ID','M4_pred']].rename(columns={'M4_pred':'ref_pred'})
    for template in ['P1','P2']:
        for pname in PERMS[target]:
            d=all_runs[(target,template,pname)].copy(); lo,hi=ci(d,'M4_pred',labels)
            metric_rows.append({
                'target':target,'template':template,'permutation':pname,'n':len(d),
                'macro_f1':f1_score(d.y_true,d.M4_pred,average='macro'),
                'accuracy':accuracy_score(d.y_true,d.M4_pred),'ci_low':lo,'ci_high':hi,
            })
            for lab,n in d.M4_pred.astype(int).value_counts().sort_index().items():
                dist_rows.append({'target':target,'template':template,'permutation':pname,
                                  'predicted_label':int(lab),'n':int(n),'proportion':float(n/len(d))})
            a=d[['Case_ID','M4_pred']].merge(ref,on='Case_ID',validate='one_to_one')
            agreement_rows.append({
                'target':target,'template':template,'permutation':pname,
                'agreement_with_P1_identity':float((a.M4_pred.astype(int)==a.ref_pred.astype(int)).mean()),
                'kappa_with_P1_identity':float(cohen_kappa_score(a.ref_pred.astype(int),a.M4_pred.astype(int))),
            })
metrics=pd.DataFrame(metric_rows); distributions=pd.DataFrame(dist_rows); agreements=pd.DataFrame(agreement_rows)
metrics.to_csv(OUT_DIR/'P2_81_M4_PROTOCOL_VARIANT_METRICS.csv',index=False)
distributions.to_csv(OUT_DIR/'P2_81_M4_PROTOCOL_CLASS_DISTRIBUTIONS.csv',index=False)
agreements.to_csv(OUT_DIR/'P2_81_M4_PROTOCOL_AGREEMENT.csv',index=False)
print(metrics.to_string(index=False)); print(agreements.to_string(index=False))

summary=[]
for target,g in metrics.groupby('target'):
    vals=g.macro_f1.to_numpy(float)
    p1=float(g[(g.template=='P1')&(g.permutation=='identity')].macro_f1.iloc[0])
    p2=float(g[(g.template=='P2')&(g.permutation=='identity')].macro_f1.iloc[0])
    summary.append({
        'target':target,'reference_P1_identity_macro_f1':p1,
        'P2_identity_minus_P1_identity':p2-p1,
        'protocol_mean_macro_f1':float(vals.mean()),'protocol_sd_macro_f1':float(vals.std(ddof=1)),
        'protocol_min_macro_f1':float(vals.min()),'protocol_max_macro_f1':float(vals.max()),
        'protocol_range_macro_f1':float(vals.max()-vals.min()),'n_variants':len(vals),
    })
summary=pd.DataFrame(summary)
summary.to_csv(OUT_DIR/'P2_81_M4_PROTOCOL_SENSITIVITY_SUMMARY.csv',index=False)
print(summary.to_string(index=False))

# Reproduction check against an existing reference M4 run, when available.
checks=[]
for target in ['T1','T2','T3','T4']:
    candidates=[
        RESULTS_DIR/'M4_FINAL_V3'/f'OOF_M4_FINALV3_{target}_C.csv',
        RESULTS_DIR/'M4_FORCED_CHOICE_V2'/f'OOF_M4_FORCED_{target}_C.csv',
        RESULTS_DIR/f'OOF_M4_FORCED_{target}_C.csv',
    ]
    p=next((x for x in candidates if x.exists()),None)
    if p:
        old=pd.read_csv(p)[['Case_ID','M4_pred']].rename(columns={'M4_pred':'old_pred'})
        new=all_runs[(target,'P1','identity')][['Case_ID','M4_pred']].merge(old,on='Case_ID')
        checks.append({'target':target,'comparison_file':str(p),
                       'prediction_match_rate':float((new.M4_pred.astype(int)==new.old_pred.astype(int)).mean())})
if checks:
    pd.DataFrame(checks).to_csv(OUT_DIR/'P2_81_M4_REFERENCE_REPRODUCTION_CHECK.csv',index=False)
    print(pd.DataFrame(checks).to_string(index=False))

manifest={
    'protocol':'P2_81_M4_PROTOCOL_SENSITIVITY_v1.0','model_id':MODEL_ID,'model_revision':MODEL_REVISION,
    'condition':'C','templates':['P1','P2'],'permutations':PERMS,'batch_size':BATCH_SIZE,
    'gold_sha256':sha256(INPUT_DIR/GOLD_NAME),'base_sha256':sha256(INPUT_DIR/BASE_NAME),
    'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'transformers':transformers.__version__,
}
(OUT_DIR/'P2_81_M4_PROTOCOL_SENSITIVITY_MANIFEST.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
zip_path=PROJECT_DIR/'P2_81_M4_PROTOCOL_SENSITIVITY_RESULTS.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob('*'):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT_DIR))
print('SEND THIS FILE BACK:',zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
